# Fraud Detection Model Training, Comparison, and Threshold Tuning

## Objective
This notebook strengthens the modelling stage by moving beyond a single default model. It selects the strongest fraud features, compares multiple algorithms, chooses the best-performing model, and tunes an alert-review threshold for fraud investigation workflows.


In [ ]:
# Import the core libraries used for analysis, modelling evidence, and visualisation.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Reuse the production pipeline so notebook results match the deployed app.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT / 'src'))

from fraud_pipeline import load_transactions, train_top_feature_model, TARGET


## Load Transactions
The transaction dataset contains the target label `fraud_label`. The reusable loader also validates expected binary fields and parses transaction timestamps.


In [ ]:
# Load transactions through the reusable project pipeline.
transactions = load_transactions(REPO_ROOT)

print('Transactions shape:', transactions.shape)
print('Fraud rate:', f"{transactions[TARGET].mean():.2%}")
display(transactions.head())


## Top-10 Feature Selection
The reviewer requested top features using SHAP or mutual information. This project uses mutual information at the selection stage because it is clear and direct: it estimates how much each feature reduces uncertainty about the fraud label.


In [ ]:
# Train the modelling pipeline.
# Internally this ranks features, selects the top 10, compares models, and tunes the review threshold.
bundle = train_top_feature_model(transactions, top_n=10)

print('Training rows:', bundle.train_rows)
print('Testing rows:', bundle.test_rows)
print('Selected top features:')
for rank, feature in enumerate(bundle.top_features, start=1):
    print(f'{rank}. {feature}')


In [ ]:
# Display the top ranked features and their mutual-information scores.
display(bundle.feature_importance.head(10))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=bundle.feature_importance.head(10),
    x='importance',
    y='feature',
    color='#2f6f73',
)
plt.title('Top 10 Fraud Features by Mutual Information')
plt.xlabel('Mutual information score')
plt.ylabel('Feature')
plt.show()


## Model Comparison
A stronger portfolio project should not train only one model without justification. Here, multiple algorithms are compared on the same selected top-10 features. ROC-AUC is the primary ranking metric because fraud detection is a prioritisation problem: analysts need risky transactions surfaced ahead of lower-risk transactions.


In [ ]:
# Compare candidate models using ROC-AUC, precision, recall, and F1-score.
display(bundle.model_comparison)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=bundle.model_comparison,
    x='roc_auc',
    y='model',
    color='#3b82a0',
)
plt.title('Candidate Model Comparison')
plt.xlabel('ROC-AUC')
plt.ylabel('Model')
plt.xlim(0, 1)
plt.show()

print('Selected model:', bundle.model_name)


## Threshold Tuning
Fraud teams rarely use the default probability threshold of 0.50. A lower review threshold may catch more fraud, but it also increases investigation volume. This section evaluates thresholds and selects a practical alert threshold based on F1-score while keeping the queue manageable.


In [ ]:
# Review the threshold tuning table.
display(bundle.threshold_table.sort_values('f1', ascending=False).head(10))

plt.figure(figsize=(10, 5))
plt.plot(bundle.threshold_table['threshold'], bundle.threshold_table['precision'], marker='o', label='Precision')
plt.plot(bundle.threshold_table['threshold'], bundle.threshold_table['recall'], marker='o', label='Recall')
plt.plot(bundle.threshold_table['threshold'], bundle.threshold_table['f1'], marker='o', label='F1-score')
plt.axvline(bundle.review_threshold, color='red', linestyle='--', label=f'Selected threshold: {bundle.review_threshold:.2f}')
plt.title('Fraud Alert Threshold Tuning')
plt.xlabel('Fraud probability threshold')
plt.ylabel('Score')
plt.legend()
plt.show()


## Final Model Evaluation
The final model is evaluated using the selected review threshold. This is more realistic than only reporting default classifier output because fraud alert systems are tuned for operational trade-offs.


In [ ]:
# Summarise final top-feature model performance.
print('Selected model:', bundle.model_name)
print('Selected review threshold:', round(bundle.review_threshold, 2))
print('ROC-AUC:', round(bundle.metrics['roc_auc'], 4))
print('Precision:', round(bundle.metrics['precision'], 4))
print('Recall:', round(bundle.metrics['recall'], 4))
print('F1-score:', round(bundle.metrics['f1'], 4))

report = pd.DataFrame(bundle.metrics['classification_report']).T
display(report)


## Modelling Insight
The final model intentionally uses only the top 10 selected features. This makes the system easier to explain, keeps the dashboard focused on the strongest fraud drivers, and gives analysts a practical threshold for prioritising review queues.
